# Améliorer les images de télescopes avec l'IA 🔭

*Par : Gabriel Missael Barco, Nicolas Payot, Auriane Thilloy, Olivia Pereira, Noé Dia, Guillaume Payeur*

<a href="https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Diffusion_Simulated_Galaxy_Pipeline_fr_v2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"/></a> <a href="https://github.com/GabrielMissael/super-resolution-workshop"><img src="https://img.shields.io/badge/View_on-GitHub-black?logo=github" alt="Voir sur GitHub"/></a>

<img src="https://i.imgur.com/i8Rl4ef.png" alt="Introduction Banner" height="250"/>

**Bienvenue!** Aujourd'hui, nous allons mélanger **Astronomie** et **Intelligence Artificielle**.

Les vrais télescopes ne sont pas parfaits. Les images qu'ils prennent sont souvent floues, pixélisées et bruitées. Dans cet atelier, nous allons apprendre à les réparer!

**Le plan :**

1.  **Briser l'image :** Nous allons simuler un "mauvais" télescope pour voir comment il gâche une image parfaite de galaxie.
2.  **Réparer l'image :** Nous allons utiliser un **modèle d'IA** intelligent qui a appris à quoi ressemblent les galaxies.
3.  **Combiner les deux :** Nous allons apprendre à l'IA à regarder une image dégradée et à reconstruire la galaxie nette qui s'y cache.
4.  **Le Mystère :** Pouvez-vous identifier un "Objet Mystère" à partir d'une tache floue? 👀
5.  **Le Concours :** Transformez-*vous* (ou n'importe quel objet) en galaxie! Les images les plus créatives gagneront un prix! 🏆

## 1. Préparation du laboratoire 🤖

D'abord, nous devons charger nos outils. Ce code prépare notre "télescope virtuel" et charge le **Cerveau de l'IA** (un modèle qui a étudié des milliers d'images de galaxies).

Vous n'avez pas besoin de comprendre le code dans cette cellule spécifique; voyez cela comme le démarrage du moteur de notre voiture! 🚗

In [ ]:
# @title
print("Téléchargement des données... 💾")
!git clone --quiet https://github.com/GabrielMissael/super-resolution-workshop

print("Récupération du code... 🤖")
!pip3 install --quiet git+https://github.com/AlexandreAdam/score_models.git@dev

import sys
sys.path.append("super-resolution-workshop")
from src.diffusion_sampling.diffusion_sampling import *
from src.diffusion_sampling.telescope_app import launch_telescope_app
from pathlib import Path
from huggingface_hub import snapshot_download
from score_models import ScoreModel
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import gradio as gr
import numpy as np
import matplotlib.cm as cm

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

MODEL_DIR = Path("model/galaxy_prior")

if not MODEL_DIR.exists():
    print("Téléchargement de l'a priori de galaxie depuis Hugging Face... 🧐")
    snapshot_download(
        repo_id="GMissaelBarco/galaxy-prior",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
        tqdm_class=None,
    )

model = ScoreModel(path=str(MODEL_DIR)).to(DEVICE)
model.load()
model.eval()
print("Modèle chargé ✅")

galaxies = torch.load("super-resolution-workshop/data/galaxies.pt", map_location=DEVICE)

show_grid(galaxies, title="Exemples de galaxies nettes")

%config InlineBackend.figure_format = 'retina'

## 2. Effet de télescope #1 : Le Flou (PSF) 👓

<img src="https://i.imgur.com/saE85st.png" alt="PSF example" height="150"/>

Les vrais télescopes ne sont pas parfaits. À cause de l'atmosphère et de l'optique, la lumière est étalée. Les astronomes appellent cela la **Fonction d'étalement du point (PSF)**, mais vous pouvez simplement imaginer que c'est comme regarder à travers de **mauvaises lunettes**.

**Essayez par vous-même :**

* Déplacez le curseur `Sigma PSF`.
* Regardez comment les détails des bras spiraux fondent à mesure que le flou devient plus fort.

In [ ]:
# @title
sigma_psf_slider_explore = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=5.0,
    step=0.01,
    description="Sigma PSF :",
    layout=widgets.Layout(width="800px"),
)

out_psf = widgets.Output()

def update_psf_explore(change=None):
    with out_psf:
        clear_output(wait=True)
        psf_images = psf_on_image(galaxies, sigma=float(sigma_psf_slider_explore.value))
        show_grid(psf_images, title="Effet de la PSF (flou seulement)")

sigma_psf_slider_explore.observe(update_psf_explore, names="value")
display(sigma_psf_slider_explore)
update_psf_explore()
display(out_psf)

## 3. Effet de télescope #2 : Basse Résolution 🧱

<img src="https://i.imgur.com/KC6LMZQ.png" alt="Downsampling example" height="150"/>

Les caméras numériques utilisent des **pixels**. Si nous n'avons pas assez de pixels, l'image semble faite de blocs (comme dans Minecraft ou les vieux jeux vidéo). On perd les petits détails.

**Essayez par vous-même :**

* Baissez le curseur `Pixels réduits à :`.
* Voyez comment la galaxie se transforme en une grille de blocs.

In [ ]:
# @title
pixels_downsample_slider_explore = widgets.IntSlider(
    value=64,
    min=8,
    max=64,
    step=1,
    description="Pixels réduits à :",
    layout=widgets.Layout(width="800px"),
)

out_downsample = widgets.Output()

def update_downsample_explore(change=None):
    with out_downsample:
        clear_output(wait=True)
        downsampled = downsample_img(galaxies, size=int(pixels_downsample_slider_explore.value))
        show_grid(downsampled, title="Effet du sous-échantillonnage (résolution seulement)")

pixels_downsample_slider_explore.observe(update_downsample_explore, names="value")
display(pixels_downsample_slider_explore)
update_downsample_explore()
display(out_downsample)

## 4. Effet de télescope #3 : Le Bruit (Statique) 🌧️

<img src="https://i.imgur.com/ZBLnBF1.png" alt="Noise example" height="150"/>

Les détecteurs ne sont pas parfaits non plus! Parfois, ils enregistrent des signaux aléatoires, comme de la "statique" (la neige) sur une vieille télé ou du grain sur une photo prise avec peu de lumière.

**Essayez par vous-même :**

* Augmentez le curseur `Bruit σ :`.
* Remarquez à quel point il devient difficile de voir la forme de la galaxie quand la "neige" prend le dessus.

In [ ]:
# @title
sigma_noise_slider_explore = widgets.FloatSlider(
    value=0.00,
    min=0.0,
    max=0.5,
    step=0.0005,
    description="Bruit σ :",
    layout=widgets.Layout(width="800px"),
)

out_noise = widgets.Output()

def update_noise_explore(change=None):
    with out_noise:
        clear_output(wait=True)
        noisy = add_gaussian_noise(galaxies, sigma=float(sigma_noise_slider_explore.value))
        show_grid(noisy, title="Effet du bruit gaussien")

sigma_noise_slider_explore.observe(update_noise_explore, names="value")
display(sigma_noise_slider_explore)
update_noise_explore()
display(out_noise)

## 5. Construire un télescope réaliste 🧪

<img src="https://i.imgur.com/tJAzpVK.png" alt="Pipeline example" height="150"/>

Dans le monde réel, **les trois problèmes surviennent en même temps**. Un télescope rend l'image floue, les pixels la rendent carrée (pixélisée), et la caméra ajoute du bruit.

**À votre tour :**

1.  Utilisez les curseurs ci-dessous pour gâcher une image de galaxie parfaite.
2.  **Choisissez une galaxie** que vous préférez; nous essaierons de la sauver à la prochaine étape!

Quel paramètre (Flou, Résolution ou Bruit) rend l'image la plus difficile à reconnaître?

In [ ]:
# @title
# Sliders partagés pour le modèle direct (forward model) utilisé plus tard dans l'inférence
sigma_psf_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=5.0,
    step=0.01,
    description="Sigma PSF :",
    layout=widgets.Layout(width="800px"),
)

pixels_downsample_slider = widgets.IntSlider(
    value=64,
    min=10,
    max=64,
    step=1,
    description="Pixels réduits à :",
    layout=widgets.Layout(width="800px"),
)

sigma_noise_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=0.2,
    step=0.0005,
    description="Bruit σ :",
    layout=widgets.Layout(width="800px"),
)

image_selector = widgets.ToggleButtons(
    options=[("Image 1", 0), ("Image 2", 1), ("Image 3", 2), ("Image 4", 3), ("Image 5", 4)],
    description="Sélectionner l'image :",
    layout=widgets.Layout(width="800px"),
)

out_pipeline = widgets.Output()

# Variables globales à réutiliser dans les cellules suivantes
galaxies_psf = None
galaxies_downsampled = None
galaxies_noisy = None

def update_pipeline(change=None):
    global galaxies_psf, galaxies_downsampled, galaxies_noisy

    with out_pipeline:
        clear_output(wait=True)

        sigma_psf_val = float(sigma_psf_slider.value)
        size_val = int(pixels_downsample_slider.value)
        sigma_n_val = float(sigma_noise_slider.value)

        galaxies_psf = psf_on_image(galaxies, sigma=sigma_psf_val)
        galaxies_downsampled = downsample_img(galaxies_psf, size=size_val)
        galaxies_noisy = add_gaussian_noise(galaxies_downsampled, sigma_n_val)

        selected_idx = image_selector.value
        show_grid_final(galaxies_noisy, selected_idx=selected_idx)

for w in [sigma_psf_slider, pixels_downsample_slider, sigma_noise_slider, image_selector]:
    w.observe(update_pipeline, names="value")

ui = widgets.VBox(
    [
        sigma_psf_slider,
        pixels_downsample_slider,
        sigma_noise_slider,
        image_selector,
        out_pipeline,
    ]
)

display(ui)
update_pipeline()

## 6. Le tour de "Magie" : Reconstruire la galaxie 🔄

Voici le défi : **Peut-on transformer cette tache laide et bruitée en une galaxie nette?**

Pour ce faire, nous utilisons un **Jeu de devinettes intelligent** :

1.  **La Physique :** Nous savons comment le télescope a brisé l'image (le flou et le bruit).
2.  **L'IA :** Nous avons une IA qui sait à quoi ressemblent les galaxies *nettes*.
3.  **La Reconstruction :** L'IA essaie de dessiner une galaxie qui correspond aux données dégradées que nous avons, mais qui ressemble aussi à une galaxie réaliste.

Nous appelons ces devinettes des **"Échantillons"** (*Samples*). Voyons ce que l'IA propose!

In [ ]:
# @title
# Construire l'opérateur linéaire A correspondant à la PSF actuelle + sous-échantillonnage
sigma_psf_val = float(sigma_psf_slider.value)
S_val = int(pixels_downsample_slider.value)
y_lin, A = psf_downsample_build_A(galaxies, sigma_psf=sigma_psf_val, S=S_val)


# Choisir la galaxie sélectionnée et son observation bruitée
idx = image_selector.value
y_obs = galaxies_noisy[idx]    # (S,S)
sigma_n = float(sigma_noise_slider.value)

sampler = LinearGaussianPosteriorSampler(
    observation=y_obs,   # (S,S)
    A=A,
    model=model,
    sigma_n=sigma_n,
    C=1.0,
    M=0.0,
)

samples = sampler.run(
    n_samples=4,
    steps=100,
    progress=True,
    true=galaxies[idx],       # pour le tracé optionnel de la trajectoire
    plot_trajectory=True,    # mettre à True si vous voulez la vue animée
    trajectory_stride=5,
)

print("Format des échantillons :", samples.shape)  # (4,1,Hs,Hs)


## 8. Est-ce que ça a marché? 🎨

Regardons les résultats!

* **Rangée du haut (Les devinettes) :** La première image est la **Vraie** réponse. Les images suivantes sont les meilleures estimations de l'IA. Se ressemblent-elles?
* **Rangée du milieu (La vérification) :** Si nous prenions l'estimation de l'IA et que nous la repassions dans notre mauvais télescope, est-ce que cela correspondrait aux données que nous avons observées?
* **Rangée du bas (La différence/Résidus) :** Ceci montre ce que l'IA a manqué. Si cela ressemble à de la neige aléatoire (bruit rouge et bleu), l'IA a fait un excellent travail!

In [ ]:
# @title
fig, axes = plt.subplots(3, 5, figsize=(12, 6))

# Row 1: true + posterior samples
true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0, 0].imshow(true_img, cmap="magma")
axes[0, 0].set_title("Vérité")
axes[0, 0].axis("off")

for i in range(4):
    img = img_to_show(samples[i], log_scale=True)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Échantillon {i+1}")
    axes[0, i + 1].axis("off")

# Row 2: observed + mock observations
obs_img = img_to_show(y_obs, log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observée")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples[i].view(1, sampler.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, y_obs.shape[0], y_obs.shape[1])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Simulée {i+1}")
    axes[1, i + 1].axis("off")

    # Row 3: residuals
    residual = (y_obs - mock_img[0, 0]) / sigma_n
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Résidu {i+1}")
    axes[2, i + 1].axis("off")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()

## 9. La meilleure estimation et la carte de confusion 📊

Comme l'entrée était tellement dégradée, l'IA peut être incertaine sur certains détails.

* **Moyenne a posteriori :** C'est la moyenne de toutes les devinettes de l'IA. C'est notre unique "meilleur pari".
* **Carte d'incertitude :** Elle s'illumine dans les zones où l'IA était confuse ou là où les estimations variaient beaucoup.

In [ ]:
# @title
samples_mean = samples.mean(dim=0)  # (1,Hs,Hs)
samples_std = samples.std(dim=0)    # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

img_mean = img_to_show(samples_mean[0], log_scale=True)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Moyenne a posteriori")
axes[1].axis("off")

img_std = img_to_show(samples_std[0], log_scale=False)
axes[2].imshow(img_std, cmap="magma")
axes[2].set_title("Écart-type a posteriori")
axes[2].axis("off")

true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("Image réelle")
axes[0].axis("off")

plt.tight_layout()
plt.show()

## 10. Le Défi de la Galaxie Mystère 🕵️‍♀️

Ok, l'entraînement est terminé. Passons à la vraie mission.

Les astronomes ont capturé 12 images d'un **Objet Mystère**. Le télescope était petit, les images sont floues, et nous n'avons aucune idée de ce que c'est vraiment.

**Votre mission :** Utilisez la chaîne de traitement (pipeline) que nous venons de construire pour nettoyer les données et révéler l'objet caché. Prêts?

### 10.1 Les Données : 12 Expositions Bruitées 📷

Ci-dessous se trouvent les 12 images capturées par le télescope.

Pour l'instant, elles ressemblent juste à des taches bruitées. Mais une forme très spécifique se cache à l'intérieur. Regardez attentivement la grille—pouvez-vous deviner ce qui se cache derrière le bruit?

In [ ]:
# @title
sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

# Load observations of the mystery OOD galaxy
mistery_galaxy_obs = torch.load("super-resolution-workshop/data/mistery_galaxy_obs.pt")

sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

y_lin_ood, A = psf_downsample_build_A(
    mistery_galaxy_obs,
    sigma_psf=sigma_psf_ood,
    S=res_ood,
)

fig, ax = plt.subplots(2, 6, figsize=(12, 4))
for i in range(12):
    img = img_to_show(mistery_galaxy_obs[i], log_scale=False)
    ax[i // 6, i % 6].imshow(img, cmap="magma")
    ax[i // 6, i % 6].set_title(f"Observation {i+1}")
    ax[i // 6, i % 6].axis("off")
plt.tight_layout()
plt.show()

### 10.2 Demander à l'IA 🎲

Maintenant, lançons la simulation!

1.  Nous prenons les **12 images bruitées**.
2.  Nous utilisons le **Cerveau de l'IA** (A priori) pour s'assurer que le résultat a l'air réel.
3.  Nous générons **4 échantillons** (devinettes) de ce qu'est le véritable objet.

Voyons ce qu'elle trouve...

In [ ]:
# @title
sampler_mistery = LinearGaussianPosteriorSampler(
    observation=mistery_galaxy_obs,   # (B,res_ood,res_ood)
    A=A,
    model=model,
    sigma_n=sigma_n_ood,
    C=1.0,
    M=0.0,
)

samples_mistery = sampler_mistery.run(
    n_samples=4,
    steps=200,
    progress=True,
    true=None,
    plot_trajectory=True,
    trajectory_stride=5,
)

print("Format des échantillons hors distribution (OOD) :", samples_mistery.shape)  # (4,1,Hs,Hs)

### 10.3 L'avons-nous résolu? 👀

Avant de regarder la réponse, vérifions les maths.

* **Rangée du haut :** Les devinettes de l'IA (Échantillons).
* **Rangée du milieu :** Si nous re-floutions ces devinettes, ressembleraient-elles aux données bruitées?
* **Rangée du bas :** Les résidus (la différence).

Si les résidus ressemblent à du bruit aléatoire, notre solution est scientifiquement valide!

In [ ]:
# @title
fig, axes = plt.subplots(3, 5, figsize=(12, 9))

# Top row: OOD posterior samples
axes[0, 0].axis("off")
for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Échantillon {i+1}")
    axes[0, i + 1].axis("off")

# Middle row: Mistery galaxy observations and mock observations
obs_img = img_to_show(mistery_galaxy_obs[0], log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observée")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples_mistery[i].view(1, sampler_mistery.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, mistery_galaxy_obs.shape[1], mistery_galaxy_obs.shape[2])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Simulée {i+1}")
    axes[1, i + 1].axis("off")

    residual = (mistery_galaxy_obs[0] - mock_img[0, 0]) / sigma_n_ood
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Résidu {i+1}")
    axes[2, i + 1].axis("off")

# Question mark for unknown true image
axes[0, 0].text(0.5, 0.5, "?", fontsize=40, ha="center", va="center")
axes[0, 0].set_title("Vérité (inconnue)")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()

### 10.4 Le Grand Dévoilement 🎉

Il est temps de voir la vérité!

Nous chargeons enfin l'**Image Réelle**. Surprise! Ce n'était pas une galaxie standard... c'était une **Feuille d'érable** 🇨🇦.

Regardez comment l'IA a réussi à reconstruire la tige et les pointes, même si les données n'étaient qu'une tache floue. Cela montre la puissance de combiner la Physique avec l'IA!

In [ ]:
# @title
true_mistery = torch.load(
    "super-resolution-workshop/data/true_mistery_galaxy.pt"
).to(DEVICE)

# Plot posterior samples against the true mistery galaxy
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
true_img = img_to_show(true_mistery[0], log_scale=True, min_val=0.1)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("Vraie Galaxie Mystère")
axes[0].axis("off")

for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[i + 1].imshow(img, cmap="magma")
    axes[i + 1].set_title(f"Échantillon {i+1}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

### 10.5 Comparaison Finale ✅

Voici le côte-à-côte :

1.  **Le Véritable Objet** (Feuille d'érable).
2.  **La Meilleure Estimation de l'IA** (Moyenne a posteriori).
3.  **Ce que nous avons réellement vu** (L'observation bruitée).

D'une tache granuleuse à une feuille reconnaissable en utilisant des données + un a priori appris. Vous venez d'utiliser un modèle génératif de pointe pour résoudre un vrai problème inverse! 🛰️🍁

In [ ]:
# @title
samples_mistery_mean = samples_mistery.mean(dim=0)  # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

true_img = img_to_show(true_mistery, log_scale=True, min_val=0.2)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("Image Réelle")
axes[0].axis("off")

img_mean = img_to_show(samples_mistery_mean, log_scale=True, min_val=0.2)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Moyenne a posteriori")
axes[1].axis("off")

# Stacking result
axes[2].imshow(img_to_show(mistery_galaxy_obs[0], log_scale=False), cmap="magma")
axes[2].set_title("Observée (1ère exposition)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

# 🏆 Le Défi Final : Galaxifiez-vous! 🔭

Maintenant, c'est l'heure du **Concours Créatif**!

**La Mission :**

1.  **Prenez une photo** (de votre visage, d'un dessin, d'un objet...) ou téléversez-en une.
2.  Utilisez notre "Faux Télescope" pour la ruiner avec du bruit.
3.  Utilisez l'IA pour **la reconstruire en tant que galaxie**.

**🥇 Comment Gagner :**
Nous sélectionnerons les meilleures images pour un vote final. Pour gagner, votre image doit être **Créative** et trouver le **Juste Milieu** :

* **N'utilisez pas trop de bruit :** Si l'IA ne voit *qu'une* galaxie et qu'on ne reconnait pas votre objet original, vous ne gagnerez pas.
* **N'utilisez pas trop peu de bruit :** Si cela ressemble juste à une photo normale sans style "galaxie", vous ne gagnerez pas.
* **But :** Une image qui ressemble à une galaxie cool *mais* qui ressemble clairement à vous ou à votre objet!

**📝 Comment Soumettre & Voter :**

1.  Quand vous êtes satisfait de votre résultat, **Clic Droit -> Enregistrer l'image** pour la télécharger.
2.  Cliquez sur le **lien Padlet** dans l'application ci-dessous.
3.  Cliquez sur le bouton **(+)**, écrivez votre **Nom** comme titre, et téléversez votre image.
4.  **Regardez le tableau (ou l'écran)** pour le mot de passe—vous en aurez besoin pour entrer dans la zone de vote et "liker" vos images préférées!
5.  Si vous êtes bloqué, levez la main! Le personnel et les professeurs sont là pour aider.

**Prêts? Exécutez le code ci-dessous et cliquez sur le lien public (terminant par `gradio.live`) pour jouer!**

In [ ]:
# @title
launch_telescope_app(
    model=model,
    LinearGaussianPosteriorSampler=LinearGaussianPosteriorSampler,
    psf_downsample_build_A=psf_downsample_build_A,
    share=True,   # bon pour Colab
    debug=False,
)